In [0]:
import re
from collections import defaultdict

bronze = (spark.table("workspace.privacylens.bronze_docs")
          .orderBy("law", "page_num").collect())

HEADER_PATTERNS = {
    "GDPR": [r"Official Journal of the European Union"],
    "DPDP": [r"THE GAZETTE OF INDIA EXTRAORDINARY"],
}

def clean_page(law, text):
    lines = text.splitlines()
    keep = [l for l in lines
            if not any(re.search(p, l) for p in HEADER_PATTERNS[law])]
    return "\n".join(keep)

pages = defaultdict(list)
for r in bronze:
    pages[r.law].append((r.page_num, clean_page(r.law, r.text)))

def full_text(law):
    """Join pages into one string and remember where each page starts."""
    parts, offsets, pos = [], [], 0
    for page_num, text in pages[law]:
        offsets.append((pos, page_num))
        parts.append(text)
        pos += len(text) + 1
    return "\n".join(parts), offsets

def page_at(offsets, pos):
    page = offsets[0][1]
    for start, p in offsets:
        if start <= pos:
            page = p
        else:
            break
    return page

In [0]:
SECTION_RE = {
    "GDPR": re.compile(r"^Article (\d+)\s*$", re.M),
    "DPDP": re.compile(r"^\s*(\d+)\.\s", re.M),
}

def split_sections(law):
    text, offsets = full_text(law)
    sections = []

    # DPDP: keep the penalties Schedule as its own section
    schedule = None
    if law == "DPDP":
        heads = list(re.finditer(r"THE\s+SCHEDULE", text))
        if heads:
            sched_page = page_at(offsets, heads[-1].start())
            cut = next(start for start, p in offsets if p == sched_page)
            schedule = (text[cut:].strip(), sched_page)
            text = text[:cut]

    # Accept only sequential numbers: 1, 2, 3, ...
    matches, expected = [], 1
    for m in SECTION_RE[law].finditer(text):
        if int(m.group(1)) == expected:
            matches.append((expected, m.start()))
            expected += 1

    for i, (num, start) in enumerate(matches):
        end = matches[i + 1][1] if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        title = ""
        if law == "GDPR":
            lines = body.split("\n")
            title = lines[1].strip() if len(lines) > 1 else ""
        sections.append((str(num), title, body, page_at(offsets, start)))

    if schedule:
        sections.append(("Schedule", "Penalties", schedule[0], schedule[1]))
    return sections

In [0]:
from pyspark.sql import Row

MAX_CHARS = 2000

def sub_chunks(text, max_chars=MAX_CHARS):
    if len(text) <= max_chars:
        return [text]
    chunks, cur = [], ""
    for line in text.split("\n"):
        if len(cur) + len(line) + 1 > max_chars and cur:
            chunks.append(cur.strip())
            cur = ""
        cur += line + "\n"
    if cur.strip():
        chunks.append(cur.strip())
    return chunks

rows = []
for law in ["GDPR", "DPDP"]:
    unit = "Article" if law == "GDPR" else "Section"
    for num, title, body, page_start in split_sections(law):
        label = f"{law} {unit} {num}" if num != "Schedule" else f"{law} Schedule"
        if title:
            label += f" – {title}"
        for part, chunk in enumerate(sub_chunks(body), start=1):
            rows.append(Row(
                chunk_id=f"{law}_{num}_{part}",
                law=law,
                section_num=num,
                title=title,
                part=part,
                page_start=page_start,
                citation=label,
                chunk_text=f"{label}\n\n{chunk}",
                char_len=len(chunk),
            ))

silver = spark.createDataFrame(rows)
silver.write.mode("overwrite").saveAsTable("workspace.privacylens.silver_chunks")
print(f"Wrote {silver.count()} chunks")

In [0]:
spark.sql("""
ALTER TABLE workspace.privacylens.silver_chunks
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

spark.sql("""
COMMENT ON TABLE workspace.privacylens.silver_chunks IS
'Silver: GDPR articles and DPDP sections, header-cleaned, split by legal unit, sub-chunked to 2000 chars, with citation metadata'
""")

In [0]:
counts = {
    r.law: r.sections
    for r in spark.sql("""
        SELECT law, COUNT(DISTINCT section_num) AS sections
        FROM workspace.privacylens.silver_chunks
        GROUP BY law
    """).collect()
}

assert counts.get("GDPR") == 99, f"Expected 99 GDPR articles, got {counts.get('GDPR')}"
assert counts.get("DPDP") == 45, f"Expected 45 DPDP sections, got {counts.get('DPDP')}"
print("Quality check passed:", counts)